# Clase 030 — Strings vectorizados

**Parte 0** · VanderPlas cap. 3 § 3.11.

> 🎯 Limpiar texto sin `apply(lambda)`. El accessor `.str` es vectorizado y NaN-aware.

> ⏱️ ~60 min

## ⚙️ Setup

In [ ]:
import pandas as pd
import numpy as np

## 1️⃣ El accessor `.str`

Pandas expone métodos string vectorizados via `.str`:

```python
s.str.lower()
s.str.strip()
s.str.replace('a', 'b')
s.str.contains('patron', regex=True)
s.str.extract(r'(\d+)')
s.str.split(',', expand=True)
s.str.len()
```

**Ventaja sobre `apply(lambda)`**: vectorizado (5–10× más rápido) y maneja NaN automáticamente.

In [ ]:
emails = pd.Series([' Ana@Example.com', 'BOB@gmail.com  ', np.nan, 'cris@FOO.io', 'dan@example.com'])
print('original:')
print(emails)

limpio = emails.str.lower().str.strip()
print('\nnormalizado:')
print(limpio)
print('\nNaN se preserva sin error.')

## 2️⃣ Regex con `.str.extract`

Pattern con grupos `()` → DataFrame con una columna por grupo:

In [ ]:
dominios = limpio.str.extract(r'@(?P<dominio>[\w\.]+)$')
print(dominios)

# Combinar con el original
print('\ncombinado:')
print(pd.concat([limpio.rename('email'), dominios], axis=1))

## 3️⃣ `.str.contains` + regex para filtros

In [ ]:
df = pd.DataFrame({
    'email': ['ana@example.com', 'bob@gmail.com', 'cris@empresa.es', 'dan@yahoo.com'],
    'desc' : ['Comentario URGENTE', 'normal', 'urgente revisar', 'bug critico']
})

# Email corporativo (no es de proveedor mainstream)
mainstream = r'gmail|yahoo|hotmail|outlook'
df['corp'] = ~df['email'].str.contains(mainstream, case=False, regex=True)

# Descripción urgente (case-insensitive)
df['urgente'] = df['desc'].str.contains('urgente', case=False)

print(df)

## 4️⃣ `.str.split(expand=True)` — desnormalizar

In [ ]:
nombres = pd.Series(['Ana García', 'Bob Smith Jr', 'Cris López-Mora'])
partes = nombres.str.split(' ', n=1, expand=True)
partes.columns = ['nombre', 'apellido']
print(partes)

## 5️⃣ Dtypes string vs object

```python
pd.Series(['a','b','c'], dtype='string')  # nullable, NA-aware
pd.Series(['a','b','c'])                  # default: object (mezcla Python)
```

El dtype `'string'` es el moderno: integra con `pd.NA`, optimizaciones futuras.

## 6️⃣ `Categorical` — para baja cardinalidad

Si una columna tiene pocos valores únicos comparado al total de filas (ej: país, sexo, tipo), `Categorical` ahorra memoria masivamente y acelera `groupby`/`sort`:

In [ ]:
rng = np.random.default_rng(42)
N = 100_000
paises = rng.choice(['ES', 'CL', 'MX', 'AR', 'CO'], N)

s_obj = pd.Series(paises)
s_cat = pd.Series(paises, dtype='category')

print(f'object   : {s_obj.memory_usage(deep=True)/1024:.0f} KB')
print(f'category : {s_cat.memory_usage(deep=True)/1024:.0f} KB')
print(f'ratio    : {s_obj.memory_usage(deep=True)/s_cat.memory_usage(deep=True):.1f}×')

## ✅ Checklist

- [ ] Uso `.str.lower()`, `.str.strip()`, etc. — no `apply(lambda)`
- [ ] Sé que `.str` maneja NaN automáticamente
- [ ] Aplico regex con `.str.extract` y `.str.contains`
- [ ] Uso `.str.split(expand=True)` para desnormalizar
- [ ] Uso `Categorical` para columnas de baja cardinalidad

## 📝 Homework

Ver `README.md`. CSV contactos: normalizar email, extraer dominio, separar nombre, flag corp, Categorical país.

## 🔗 Referencias

- VanderPlas cap. 3 § 3.11
- [pandas Text](https://pandas.pydata.org/docs/user_guide/text.html)
- [pandas Categorical](https://pandas.pydata.org/docs/user_guide/categorical.html)

➡️ **Siguiente:** [031 — Series de tiempo](../031-pandas-series-de-tiempo-resampling-rolling/README.md)